In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

# ==========================================
# 1. PREPARAÇÃO DO DATASET (Estrutura ISIC)
# ==========================================
print("--- Preparando dados e simulando pipeline do ISIC ---")
# Imagens médicas de pele usam tamanhos maiores (ex: 224x224)
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 2 # Exemplo clássico binário do ISIC: Benigno vs. Maligno

# Simulando um lote de dados reais normalizados de pele para validação técnica
# Na prática, você apontaria para o diretório baixado usando:
# tf.keras.utils.image_dataset_from_directory('caminho_isic/', ...)
np.random.seed(42)
x_train = np.random.rand(120, 224, 224, 3).astype(np.float32) # Já normalizado entre 0.0 e 1.0
y_train = np.random.randint(0, NUM_CLASSES, size=(120,))
x_test = np.random.rand(30, 224, 224, 3).astype(np.float32)
y_test = np.random.randint(0, NUM_CLASSES, size=(30,))

# Convertendo rótulos para categóricos (One-Hot Encoding)
y_train_cat = keras.utils.to_categorical(y_train, NUM_CLASSES)
y_test_cat = keras.utils.to_categorical(y_test, NUM_CLASSES)


# ==========================================
# HIPERPARÂMETROS COMPARTILHADOS (Isolamento)
# ==========================================
# Garantimos a equivalência matemática solicitada usando o mesmo ecossistema de otimização
SHARED_OPTIMIZER = optimizers.Adam(learning_rate=1e-4)
SHARED_LOSS = "categorical_crossentropy"
SHARED_METRICS = ["accuracy"]
EPOCHS = 5


# ==========================================
# 2. REDE 1: CNN CUSTOMIZADA (Do zero)
# ==========================================
print("\n--- Construindo CNN Customizada Direta ---")
cnn_custom = models.Sequential([
    # Camada Convolucional + Pooling 1
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    layers.MaxPooling2D((2, 2)),

    # Camada Convolucional + Pooling 2
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Camada Convolucional + Pooling 3
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Camada de Achatamento e Densas
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5), # Regularização contra overfitting em imagens médicas
    layers.Dense(NUM_CLASSES, activation='softmax')
])

# Compilação da Rede 1
cnn_custom.compile(optimizer=SHARED_OPTIMIZER, loss=SHARED_LOSS, metrics=SHARED_METRICS)


# ==========================================
# 3. REDE 2: DEEP LEARNING (Transfer Learning com ResNet50)
# ==========================================
print("\n--- Construindo Deep CNN com ResNet50 (Transfer Learning) ---")
# Carrega a ResNet50 pré-treinada com pesos da ImageNet, descartando o topo (camadas densas originais)
base_model = keras.applications.ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Congela os pesos da base para não destruir o aprendizado prévio de bordas e texturas genéricas
base_model.trainable = False

# Conecta uma nova cabeça classificadora customizada para o ISIC
x = base_model.output
x = layers.GlobalAveragePooling2D()(x) # Reduz as dimensões substituindo o Flatten tradicional
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

resnet_transfer = models.Model(inputs=base_model.input, outputs=outputs)

# Compilação da Rede 2 (Recriando o otimizador com os mesmos parâmetros para evitar reaproveitamento de estado)
resnet_transfer.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss=SHARED_LOSS,
    metrics=SHARED_METRICS
)


# ==========================================
# 4. TREINAMENTO DOS MODELOS
# ==========================================
print("\n=== Treinando Rede 1: CNN Customizada ===")
history_custom = cnn_custom.fit(x_train, y_train_cat, validation_data=(x_test, y_test_cat), epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1)

print("\n=== Treinando Rede 2: ResNet50 Transfer Learning ===")
history_resnet = resnet_transfer.fit(x_train, y_train_cat, validation_data=(x_test, y_test_cat), epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1)


# ==========================================
# 5. PREVISÕES E AVALIAÇÃO DOS RESULTADOS
# ==========================================
print("\n--- Gerando Previsões e Avaliação dos Modelos ---")

# Previsões
pred_custom = np.argmax(cnn_custom.predict(x_test), axis=1)
pred_resnet = np.argmax(resnet_transfer.predict(x_test), axis=1)

# Relatório de Classificação
print("\n[MÉTRICAS] - CNN CUSTOMIZADA:")
print(classification_report(y_test, pred_custom, target_names=['Benigno', 'Maligno'], zero_division=0))

print("\n[MÉTRICAS] - RESNET50 TRANSFER LEARNING:")
print(classification_report(y_test, pred_resnet, target_names=['Benigno', 'Maligno'], zero_division=0))

# Matrizes de Confusão
print("\n[MATRIZ DE CONFUSÃO] - CNN Customizada:")
print(confusion_matrix(y_test, pred_custom))

print("\n[MATRIZ DE CONFUSÃO] - ResNet50:")
print(confusion_matrix(y_test, pred_resnet))


--- Preparando dados e simulando pipeline do ISIC ---

--- Construindo CNN Customizada Direta ---


/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



--- Construindo Deep CNN com ResNet50 (Transfer Learning) ---
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step

=== Treinando Rede 1: CNN Customizada ===
Epoch 1/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 18s 4s/step - accuracy: 0.5667 - loss: 0.7478 - val_accuracy: 0.4333 - val_loss: 0.8156
Epoch 2/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 29s 5s/step - accuracy: 0.5083 - loss: 0.7746 - val_accuracy: 0.5667 - val_loss: 0.6921
Epoch 3/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 16s 4s/step - accuracy: 0.4583 - loss: 0.7361 - val_accuracy: 0.4333 - val_loss: 0.7144
Epoch 4/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 20s 4s/step - accuracy: 0.5000 - loss: 0.7115 - val_accuracy: 0.4333 - val_loss: 0.7063
Epoch 5/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 21s 4s/step - accuracy: 0.5750 - loss: 0.6919 - val_accuracy: 0.4000 - val_loss: 0.6939

=== Treinando Rede 2: ResNet50 Transfer Learning ===
Epoch 1/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 44s 10s/step - accuracy: 0.5333 - loss: 1.2524 - val_accuracy: 0.5667 - val_loss: 0.8302
Epoch 2/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 40s 10s/step - 